<a href="https://colab.research.google.com/github/nicolastibata/MINE_4210_ADL_202620/blob/main/assingments/assingment_1/MINE_4210_ADL_202620_T1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Logo ADL](https://github.com/nicolastibata/MINE_4210_ADL_202620/blob/main/docs/logo.png?raw=true)


# **Taller 1: Clasificación multiclase con un MLP**

## **Integrantes: Grupo 8**
* Juan Esteban Álvarez García - 202212030
* Daniel Felipe Ortiz Vallejo - 

### **Contexto y Objetivos**

La obesidad es un problema de salud pública creciente, asociado a enfermedades cardiovasculares, diabetes y otras condiciones crónicas. Identificar el nivel de riesgo de una persona a partir de sus hábitos, sin depender de mediciones clínicas directas como el peso o la estatura, puede ser útil cuando esta información no está disponible o se busca anticipar el riesgo antes de que se manifieste físicamente. En este taller deberán estimar el nivel de obesidad de una persona a partir de variables sociodemográficas y de estilo de vida, como alimentación, actividad física, consumo de alcohol y medio de transporte.

El objetivo de este Taller es construir un perceptrón multicapa (MLP) para esta tarea, siguiendo el proceso de machine learning. Se realizará una búsqueda de hiperparámetros sobre la arquitectura de la red, variando el número de capas ocultas y el número de neuronas por capa.


| Hiperparámetro | Valores para explorar |
|-|-|
| Número de capas ocultas | 1, 2 o 3. |
| Neuronas por capa oculta | 16, 32 o 64 (mismo valor en todas las capas ocultas de una misma arquitectura). |
| Callback obligatorio | Early stopping, monitoreando val_loss, con un valor de patience justificado. |


### **Entregable**

Notebook (.ipynb) con todas las celdas ejecutadas y con todas las indicaciones anteriormente estipuladas en la sección de laboratorios en **Bloque Neón**.



### **0. Configuración inicial e importaciones**

En esta celda se importan todas las librerías necesarias para el taller:
- `numpy` y `pandas` para manejo de datos.
- `tensorflow` y `keras` para construir y entrenar el MLP.
- `sklearn` para preprocesamiento, codificación, métricas y división de datos.
- Además, se fija una semilla (`SEED = 42`) para garantizar la reproducibilidad de los resultados.

In [100]:
# Celda 0: Importaciones necesarias
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import random

# Fijar semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

### **1. Carga del conjunto de datos**

El archivo `obesidad.csv` contiene las variables sociodemográficas y de estilo de vida, junto con la variable objetivo `Nivel_obesidad`. El separador es punto y coma (`;`). Se visualizan las primeras filas para familiarizarse con la estructura.

In [101]:
# Celda 1: Cargar los datos
# El archivo está en la misma ruta
df = pd.read_csv('obesidad.csv', delimiter=';')
df.head()

,Genero,Edad,Antecedentes_familiares,Come_calorico,Frecuencia_verduras,Comidas_dia,Picoteo,Fuma,Agua_dia,Monitorea_calorias,Actividad_fisica,Tiempo_pantallas,Alcohol,Transporte,Nivel_obesidad
0,Femenino,21.0,Si,No,2,3,A veces,No,2,No,0,1,No,Transporte_Publico,Peso_Normal
1,Femenino,21.0,Si,No,3,3,A veces,Si,3,Si,3,0,A veces,Transporte_Publico,Peso_Normal
2,Masculino,23.0,Si,No,2,3,A veces,No,2,No,2,1,Frecuentemente,Transporte_Publico,Peso_Normal
3,Masculino,27.0,No,No,3,3,A veces,No,2,No,2,0,Frecuentemente,Caminando,Sobrepeso_Nivel_I
4,Masculino,22.0,No,No,2,1,A veces,No,2,No,0,0,A veces,Transporte_Publico,Sobrepeso_Nivel_II


### **2. Inspección de tipos de datos**

Se verifica el tipo de cada columna. Las variables categóricas se reconocen como `object` (strings) y las numéricas como `int64` o `float64`. Esto guiará las transformaciones posteriores.

In [102]:
# Celda 2: Convertir a tipos de datos necesarios
# Las columnas categóricas se manejarán como strings, la edad como float
# Verificamos los tipos actuales
df.dtypes

Genero                         str
Edad                       float64
Antecedentes_familiares        str
Come_calorico                  str
Frecuencia_verduras          int64
Comidas_dia                  int64
Picoteo                        str
Fuma                           str
Agua_dia                     int64
Monitorea_calorias             str
Actividad_fisica             int64
Tiempo_pantallas             int64
Alcohol                        str
Transporte                     str
Nivel_obesidad                 str
dtype: object

### **3. Exploración de variables ordinales**

Algunas columnas son ordinales (tienen un orden lógico). Se inspeccionan sus valores únicos para definir el mapeo correcto en la siguiente celda. Las variables como `Frecuencia_verduras` ya están codificadas numéricamente, pero `Picoteo` es texto y necesita un mapeo personalizado.

In [103]:
# Celda 3: Inspeccionar valores únicos de columnas ordinales y categóricas
ordinal_cols = ['Frecuencia_verduras', 'Comidas_dia', 'Picoteo', 'Agua_dia', 'Actividad_fisica', 'Tiempo_pantallas']
for col in ordinal_cols:
    print(f"{col}: {df[col].unique()}")

Frecuencia_verduras: [2 3 1]
Comidas_dia: [3 1 4 2]
Picoteo: <StringArray>
['A veces', 'Frecuentemente', 'Siempre', 'No']
Length: 4, dtype: str
Agua_dia: [2 3 1]
Actividad_fisica: [0 3 2 1]
Tiempo_pantallas: [1 0 2]


### **4. Codificación de variables ordinales**

- `Picoteo` se mapea a valores numéricos respetando su orden: `No` → 0, `A veces` → 1, `Frecuentemente` → 2, `Siempre` → 3.
- El resto de columnas ordinales (`Frecuencia_verduras`, `Comidas_dia`, etc.) ya son numéricas, solo se convierten a tipo `int` por seguridad.
- Se verifica que no haya valores no mapeados (NaN) después de la transformación.

In [104]:
# Celda 4: Definir mapeos para variables ordinales
# Solo Picoteo tiene valores categóricos de texto; las demás ya son numéricas en el orden correcto.
map_picoteo = {'No': 0, 'A veces': 1, 'Frecuentemente': 2, 'Siempre': 3}

# Aplicar mapeo solo a Picoteo
df['Picoteo'] = df['Picoteo'].map(map_picoteo)

# Las siguientes columnas ya son numéricas, no necesitan mapeo:
# Frecuencia_verduras, Comidas_dia, Agua_dia, Actividad_fisica, Tiempo_pantallas
# (Asegurarse de que sean de tipo entero)
for col in ['Frecuencia_verduras', 'Comidas_dia', 'Agua_dia', 'Actividad_fisica', 'Tiempo_pantallas']:
    df[col] = df[col].astype(int)  # o float, pero son enteros

# Verificar que no haya NaN después del mapeo
print("NaN en Picoteo después del mapeo:", df['Picoteo'].isna().sum())
print("NaN en otras ordinales:", df[['Frecuencia_verduras', 'Comidas_dia', 'Agua_dia', 'Actividad_fisica', 'Tiempo_pantallas']].isna().sum().sum())

NaN en Picoteo después del mapeo: 0
NaN en otras ordinales: 0


### **5. Codificación one-hot de variables nominales**

Las variables nominales (sin orden) se convierten en variables dummy mediante one-hot encoding. Esto evita asignar un orden inexistente. La función `pd.get_dummies` crea una columna por cada categoría.

In [105]:
# Celda 5: Codificar variables nominales con one-hot
nominal_cols = ['Genero', 'Antecedentes_familiares', 'Come_calorico', 'Fuma', 'Monitorea_calorias', 'Alcohol', 'Transporte']
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=False)

### **6. Separación de características (X) y objetivo (y)**

- Se extrae la columna `Nivel_obesidad` como variable objetivo `y`.
- El resto de columnas forman las características `X`.
- La variable objetivo se codifica con `LabelEncoder` para convertir las etiquetas de clase (strings) en números enteros (0,1,2,...). Se guardan los nombres de las clases para su uso posterior en reportes.

In [106]:
# Celda 6: Separar X y y
# La variable objetivo es 'Nivel_obesidad'
y = df_encoded['Nivel_obesidad']
X = df_encoded.drop(columns=['Nivel_obesidad'])

# Codificar la variable objetivo con LabelEncoder (para tener 0..6)
le = LabelEncoder()
y_encoded = le.fit_transform(y)
# Guardar las clases originales para interpretación posterior
class_names = le.classes_

### **7. Análisis del balance de clases**

Se revisa la distribución de las 7 clases de obesidad. Esto es importante para saber si el conjunto está desbalanceado, lo que podría afectar el entrenamiento y la evaluación. Se muestran frecuencias absolutas y porcentajes.

In [107]:
# Celda 7: Balance de clases de la variable objetivo
print("Distribución de clases en Nivel_obesidad:")
print(pd.Series(y_encoded).value_counts().sort_index())
print("\nPorcentajes:")
print(pd.Series(y_encoded).value_counts(normalize=True).sort_index() * 100)

Distribución de clases en Nivel_obesidad:
0    351
1    297
2    324
3    267
4    282
5    276
6    290
Name: count, dtype: int64

Porcentajes:
0    16.818400
1    14.230954
2    15.524677
3    12.793483
4    13.512218
5    13.224724
6    13.895544
Name: proportion, dtype: float64


### **8. División en conjuntos de entrenamiento, validación y prueba**

Se usa una división estratificada (70% entrenamiento, 20% validación, 10% prueba) para mantener la proporción de clases en cada subconjunto. El conjunto de prueba no se utilizará durante la búsqueda de hiperparámetros, solo al final para una evaluación imparcial.

In [108]:
# Celda 8: Separación 70/20/10
# Primero separamos train (70%) y temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.3, random_state=SEED, stratify=y_encoded
)

# Luego separamos temp en validation (20% del total) y test (10% del total)
# Como temp es 30%, validation debe ser 20/30 = 2/3 de temp, test 10/30 = 1/3 de temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=1/3, random_state=SEED, stratify=y_temp
)

print(f"Tamaño train: {X_train.shape[0]}")
print(f"Tamaño validation: {X_val.shape[0]}")
print(f"Tamaño test: {X_test.shape[0]} (no se usará)")

Tamaño train: 1460
Tamaño validation: 418
Tamaño test: 209 (no se usará)


### **9. Normalización de características numéricas**

- Se seleccionan las columnas numéricas (incluyendo las ordinales ya codificadas y la edad).
- Se descartan columnas con varianza cero (no aportan información).
- Se ajusta un `StandardScaler` sobre el conjunto de entrenamiento y se transforman entrenamiento, validación y prueba con los mismos parámetros (evitando fuga de información).
- Se verifica que no haya valores NaN o infinitos después del escalado.
- También se confirma que las etiquetas sean enteros.

In [109]:
# Celda 9: Normalización de características numéricas (incluye ordinales numéricas y Edad)
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Identificar columnas con varianza cero en train
cols_to_scale = []
for col in numeric_cols:
    if X_train[col].std() == 0:
        print(f"Columna '{col}' tiene varianza cero. No se escalará.")
    else:
        cols_to_scale.append(col)

# Escalar solo las columnas con varianza > 0
# Ajustar scaler con train y transformar train, val y test (aunque test no se use)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

if cols_to_scale:
    X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
    X_val_scaled[cols_to_scale] = scaler.transform(X_val[cols_to_scale])
    X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])
else:
    print("No hay columnas numéricas para escalar.")

# Verificar que no haya NaN o infinitos en los datos escalados
print("\nVerificación de valores no finitos en datos escalados:")
for dataset, name in [(X_train_scaled, 'train'), (X_val_scaled, 'val'), (X_test_scaled, 'test')]:
    has_nan = dataset.isna().any().any()
    has_inf = np.isinf(dataset).any().any()
    print(f"  {name}: NaN = {has_nan}, Inf = {has_inf}")

# Verificar que las etiquetas son enteros
print("\nTipo de etiquetas:")
print(f"  y_train: {y_train.dtype}, valores únicos: {np.unique(y_train)}")
print(f"  y_val:   {y_val.dtype}, valores únicos: {np.unique(y_val)}")


Verificación de valores no finitos en datos escalados:
  train: NaN = False, Inf = False
  val: NaN = False, Inf = False
  test: NaN = False, Inf = False

Tipo de etiquetas:
  y_train: int64, valores únicos: [0 1 2 3 4 5 6]
  y_val:   int64, valores únicos: [0 1 2 3 4 5 6]


### **10. Función para construir el MLP**

Se define una función que crea un modelo secuencial de Keras con:
- Capa de entrada (definida mediante `Input`).
- Un número variable de capas ocultas (1, 2 o 3) y neuronas por capa (16, 32 o 64).
- Función de activación ReLU en las capas ocultas e inicialización He normal.
- Capa de salida con activación softmax para clasificación multiclase.
- Optimizador Adam con tasa de aprendizaje configurable.
- Función de pérdida `sparse_categorical_crossentropy` (adecuada para etiquetas enteras).
- Métrica de precisión.

In [110]:
# Celda 10: Definir la función para crear el modelo (usando Input)

def build_model(hidden_layers, neurons, input_dim, output_dim, lr=0.001):
    model = Sequential()
    # Capa de entrada
    model.add(Input(shape=(input_dim,)))
    for _ in range(hidden_layers):
        # Capas ocultas
        model.add(Dense(neurons, activation='relu', kernel_initializer='he_normal'))
    # Capa de salida
    model.add(Dense(output_dim, activation='softmax'))
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

### **11. Búsqueda de hiperparámetros (grid search)**

Se exploran las 9 combinaciones de:
- Número de capas ocultas: 1, 2, 3
- Neuronas por capa: 16, 32, 64

Para cada combinación:
- Se construye el modelo con la función `build_model`.
- Se entrena durante un máximo de 100 épocas con un `EarlyStopping` que monitorea `val_loss` y tiene `patience=10` (se detiene si la pérdida de validación no mejora durante 10 épocas).
- Se registra la mejor pérdida de validación obtenida.

Se eligió 10 como valor para la paciencia porque es un número conservador y ampliamente usado en la práctica. Permite evitar paradas prematuras y sobreajuste, además de equilibrar el rendimiento y el tiempo de cómputo.

In [111]:
# Celda 11: Búsqueda de hiperparámetros (9 configuraciones)
input_dim = X_train_scaled.shape[1]
output_dim = len(np.unique(y_train))

# Lista de configuraciones
hidden_layers_options = [1, 2, 3]
neurons_options = [16, 32, 64]

best_val_loss = float('inf')
best_config = None
best_model = None

for layers in hidden_layers_options:
    for neurons in neurons_options:
        print(f"\nEntrenando modelo con {layers} capas ocultas y {neurons} neuronas por capa (lr=0.001)...")
        
        model = build_model(layers, neurons, input_dim, output_dim, lr=0.001)
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        
        history = model.fit(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            epochs=300, # número suficiente para que early stopping actúe
            batch_size=32,
            callbacks=[early_stop],
            verbose=1
        )
        
        # Obtener el mejor val_loss (después de early stopping, el modelo tiene los mejores pesos)
        # El historial contiene el val_loss de cada época, tomamos el mínimo        
        min_val_loss = min(history.history['val_loss'])
        
        print(f"  Mejor val_loss: {min_val_loss:.4f}")
        
        if min_val_loss < best_val_loss:
            best_val_loss = min_val_loss
            best_config = (layers, neurons)
            best_model = model # guardamos el modelo ya entrenado con los mejores pesos

print(f"\nMejor configuración: {best_config[0]} capas ocultas, {best_config[1]} neuronas, con val_loss = {best_val_loss:.4f}")


Entrenando modelo con 1 capas ocultas y 16 neuronas por capa (lr=0.001)...
Epoch 1/300
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1432 - loss: 2.0351 - val_accuracy: 0.1866 - val_loss: 1.9431
Epoch 2/300
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2349 - loss: 1.9014 - val_accuracy: 0.2847 - val_loss: 1.8406
Epoch 3/300
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2986 - loss: 1.8030 - val_accuracy: 0.3325 - val_loss: 1.7558
Epoch 4/300
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3390 - loss: 1.7163 - val_accuracy: 0.3493 - val_loss: 1.6773
Epoch 5/300
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3856 - loss: 1.6360 - val_accuracy: 0.3804 - val_loss: 1.6036
Epoch 6/300
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4288 - loss: 1.5623 - val_accuracy: 0.4306 - val_loss: 1.5333
Epoch 7/300
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4678 - loss: 1.4947 - val_accuracy: 0.4545 - val_loss: 1.4664
Epoch 8/300
46/46 ━━━━━━━━━━━━━━━━━

### **12. Evaluación final del mejor modelo**

Con la mejor configuración encontrada, se evalúa el modelo sobre el conjunto de validación (no se usa el conjunto de prueba en esta etapa para evitar sesgo en la selección del modelo). Se calculan:
- Matriz de confusión.
- Reporte de clasificación con precisión, recall y F1-score por clase.
- Exactitud (accuracy) y error.

Estas métricas permiten juzgar el desempeño del modelo y detectar posibles clases con bajo rendimiento.

In [112]:
# Celda 12: Evaluación final del mejor modelo en el conjunto de validation
# (No se usa test aún)
y_pred_probs = best_model.predict(X_val_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Matriz de confusión
cm = confusion_matrix(y_val, y_pred)
print("Matriz de confusión (en validation):")
print(cm)
print("\nClases (orden):", class_names)

# Métricas: accuracy, recall, precision, f1-score
print("\nReporte de clasificación (validation):")
print(classification_report(y_val, y_pred, target_names=class_names, digits=4))

acc = np.mean(y_pred == y_val)
print(f"\nAccuracy: {acc:.4f}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Matriz de confusión (en validation):
[[61  2  1  0  1  2  4]
 [ 1 55  0  0  0  3  0]
 [ 0  0 64  0  0  0  1]
 [ 1  0  0 42  9  1  0]
 [ 2  0  1 15 30  7  2]
 [10  1  1  0  4 36  3]
 [18  5  1  0  4  3 27]]

Clases (orden): ['Obesidad_Tipo_I' 'Obesidad_Tipo_II' 'Obesidad_Tipo_III'
 'Peso_Insuficiente' 'Peso_Normal' 'Sobrepeso_Nivel_I'
 'Sobrepeso_Nivel_II']

Reporte de clasificación (validation):
                    precision    recall  f1-score   support

   Obesidad_Tipo_I     0.6559    0.8592    0.7439        71
  Obesidad_Tipo_II     0.8730    0.9322    0.9016        59
 Obesidad_Tipo_III     0.9412    0.9846    0.9624        65
 Peso_Insuficiente     0.7368    0.7925    0.7636        53
       Peso_Normal     0.6250    0.5263    0.5714        57
 Sobrepeso_Nivel_I     0.6923    0.6545    0.6729        55
Sobrepeso_Nivel_II     0.7297    0.4655    0.5684        58

          accuracy                         0.7536       418
         macro avg 

# TODO Daniel
* Gráficas Loss function
* Demás gráficas importantes
* Interpretación y conclusiones sobre validation
* Ejecutar con test
* Interpretación y conclusiones sobre test
* Veredicto sobre el modelo y sugerencias finales

### **Rúbrica**

| Criterio | Peso | Qué se evalúa |
|-|-|-|
| Preprocesamiento de datos | 15 % | Codificación adecuada de las variables categóricas, diferenciando entre nominales y ordinales escalamiento de las variables numéricas; y separación correcta de los conjuntos de entrenamiento, <br> validación y prueba, evitando fuga de información. |
| Implementación del MLP | 15 % | Arquitectura correctamente construida, con función de activación de salida y función de pérdida apropiadas para clasificación multiclase; código funcional y bien organizado. |
| Búsqueda de hiperparámetros | 25 % | Las 9 configuraciones están correctamente implementadas y entrenadas, early stopping está configurado con un valor de patience justificado. Los resultados se presentan de forma <br> clara mediante una tabla o gráfico comparativo. |
| Evaluación y métricas | 15 % | Uso de métricas apropiadas para clasificación multiclase, considerando posibles diferencias en el tamaño de las clases; evaluación final realizada correctamente sobre el conjunto de test.  |
| Interpretación y justificación | 20 % | Justificación de la arquitectura final con base en los resultados y las curvas de entrenamiento y validación; identificación de señales de sobreajuste o subajuste; conclusiones sobre el efecto <br> de la profundidad y el ancho de la red. |
| Calidad y reproducibilidad del notebook | 10 % | El notebook se ejecuta de principio a fin sin errores, está organizado en un orden lógico, no contiene código innecesario e incluye explicaciones breves en Markdown que guían la lectura. |